# SA-DBNet v2.0 — Bounding Box Detection Training (Kaggle T4)
**Architecture:** ResNet-18 + Self-Attention + Deformable FPN + Differentiable Binarization

Training on ICDAR 2015 (1000 images) with:
- Data augmentation (rotation, scale, color jitter, random crop)
- More epochs (100)
- Larger batch size (8) leveraging T4 16GB VRAM
- Fine-tuning from the 44% F1 checkpoint (or from scratch with ImageNet backbone)

In [2]:
!pip install -q pyclipper shapely

In [10]:
import os
for root, dirs, files in os.walk('/kaggle/input'):
    if 'train_images' in dirs:
        print(f'IMAGES: {root}/train_images ({len(os.listdir(os.path.join(root, "train_images")))} files)')
    if 'train_gt' in dirs:
        print(f'GT: {root}/train_gt ({len(os.listdir(os.path.join(root, "train_gt")))} files)')

import glob
pths = glob.glob('/kaggle/input/**/*.pth', recursive=True)
print(f'PTH files: {pths}')


IMAGES: /kaggle/input/datasets/kagglemodeltraining/icdar-2015-detection/icdar2015_detection/icdar2015/train_images (1000 files)
GT: /kaggle/input/datasets/kagglemodeltraining/icdar-2015-detection/icdar2015_detection/icdar2015/train_gt (1000 files)
PTH files: ['/kaggle/input/datasets/kagglemodeltraining/icdar-2015-detection/sadbnet_flawless_best.pth']


## 1. Extract ICDAR Data
**Upload `icdar2015_detection.tar.gz` as a Kaggle Dataset first!**

In [14]:
# Fix the train_det.py to load pretrained weights
import re
with open('train_det.py', 'r') as f:
    code = f.read()

# Add pretrained loading after model creation
old = "model = SADBNet().to(DEVICE)"
new = """model = SADBNet().to(DEVICE)

    # Load the 44% F1 checkpoint
    pretrained = '/kaggle/input/datasets/kagglemodeltraining/icdar-2015-detection/sadbnet_flawless_best.pth'
    if os.path.exists(pretrained):
        model.load_state_dict(torch.load(pretrained, map_location=DEVICE))
        print('Loaded pre-trained 44% F1 weights!')
    else:
        print('WARNING: pretrained not found!')"""

code = code.replace(old, new, 1)
code = code.replace('SA-DBNet v2.0 — Kaggle Training (100 Epochs)', 'SA-DBNet — Fine-Tuning from 44% F1')
with open('train_det.py', 'w') as f:
    f.write(code)
print('Fixed!')


Fixed!


## 2. Write SA-DBNet Model

In [5]:
%%writefile train_det.py
import os, time, torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader
from tqdm import tqdm
from sadbnet_model import SADBNet
from det_dataset import DetDataset

BATCH_SIZE = 8
EPOCHS = 100
LR = 5e-5
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CKPT = '/kaggle/working/checkpoints'
TRAIN_IMG = '/kaggle/input/datasets/kagglemodeltraining/icdar-2015-detection/icdar2015/train_images'
TRAIN_GT = '/kaggle/input/datasets/kagglemodeltraining/icdar-2015-detection/icdar2015/train_gt'
PRETRAINED = '/kaggle/input/datasets/kagglemodeltraining/icdar-2015-detection/sadbnet_flawless_best.pth'

class DBNetLoss(nn.Module):
    def __init__(self, alpha=1.0, beta=10.0, l1_scale=10.0):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.bce = nn.BCELoss(reduction='none')
        self.l1 = nn.L1Loss(reduction='none')
        self.l1_scale = l1_scale

    def forward(self, pred, target):
        prob, thresh, binary = pred
        t_prob = target['prob_map']
        mask = target['mask']
        t_thresh = target['thresh_map']
        t_mask = target['thresh_mask']
        bce = torch.sum(self.bce(prob, t_prob) * mask) / (torch.sum(mask) + 1e-6)
        l1 = torch.sum(self.l1(thresh, t_thresh) * t_mask) / (torch.sum(t_mask) + 1e-6) * self.l1_scale
        inter = torch.sum(binary * t_prob * mask)
        union = torch.sum(binary * mask) + torch.sum(t_prob * mask)
        dice = 1.0 - (2.0 * inter) / (union + 1e-6)
        total = bce + self.alpha * l1 + self.beta * dice
        return total, bce, l1, dice

def train():
    os.makedirs(CKPT, exist_ok=True)
    print('='*60)
    print('SA-DBNet v2.0 — Fine-Tuning from 44% F1 Checkpoint')
    print('='*60)

    ds = DetDataset(TRAIN_IMG, TRAIN_GT, augment=True)
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)

    model = SADBNet().to(DEVICE)

    # Load the 44% F1 checkpoint as starting point
    if os.path.exists(PRETRAINED):
        model.load_state_dict(torch.load(PRETRAINED, map_location=DEVICE))
        print(f'Loaded pre-trained 44% F1 weights!')
    else:
        print(f'WARNING: {PRETRAINED} not found, training from scratch!')

    if torch.cuda.device_count() > 1:
        print(f'Using {torch.cuda.device_count()} GPUs!')
        model = nn.DataParallel(model)

    params = sum(p.numel() for p in model.parameters())
    print(f'Parameters: {params/1e6:.2f}M')

    criterion = DBNetLoss()
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

    best_loss = float('inf')

    for epoch in range(EPOCHS):
        model.train()
        total_loss, total_bce, total_dice = 0, 0, 0

        pbar = tqdm(loader, desc=f'Epoch {epoch+1}/{EPOCHS}')
        for batch in pbar:
            images = batch['image'].to(DEVICE)
            target = {k: v.to(DEVICE) for k, v in batch.items() if k != 'image'}

            optimizer.zero_grad()
            pred = model(images)
            loss, bce, l1, dice = criterion(pred, target)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()

            total_loss += loss.item()
            total_bce += bce.item()
            total_dice += dice.item()
            pbar.set_postfix(loss=f'{loss.item():.3f}', bce=f'{bce.item():.4f}')

        scheduler.step()
        avg = total_loss / len(loader)
        lr = optimizer.param_groups[0]['lr']
        print(f'Epoch {epoch+1} | Loss: {avg:.4f} | BCE: {total_bce/len(loader):.4f} | Dice: {total_dice/len(loader):.4f} | LR: {lr:.6f}')

        state = model.module.state_dict() if hasattr(model, 'module') else model.state_dict()
        torch.save(state, os.path.join(CKPT, 'sadbnet_kaggle_best.pth'))
        if avg < best_loss:
            best_loss = avg
            torch.save(state, os.path.join(CKPT, 'sadbnet_kaggle_finetuned.pth'))
            print(f'  >>> New Best: {best_loss:.4f}')

    print(f'\nDone! Best Loss: {best_loss:.4f}')

if __name__ == '__main__':
    train()


Overwriting train_det.py


## 3. Write Dataset with Augmentation

In [6]:
%%writefile det_dataset.py
import os, cv2, glob, random, numpy as np, pyclipper, torch
from shapely.geometry import Polygon
from torch.utils.data import Dataset

class DetDataset(Dataset):
    def __init__(self, image_dir, gt_dir, img_size=(640, 640), shrink_ratio=0.4, augment=True):
        self.image_dir = image_dir
        self.gt_dir = gt_dir
        self.img_size = img_size
        self.shrink_ratio = shrink_ratio
        self.augment = augment
        self.image_paths = []
        for ext in ['*.jpg', '*.png', '*.JPG', '*.PNG']:
            self.image_paths.extend(glob.glob(os.path.join(image_dir, ext)))
        print(f'DetDataset: {len(self.image_paths)} images from {image_dir}')

    def __len__(self): return len(self.image_paths)

    def load_gt(self, gt_path):
        polys, tags = [], []
        if not os.path.exists(gt_path): return np.array([]), np.array([])
        with open(gt_path, 'r', encoding='utf-8-sig') as f:
            for line in f:
                parts = line.strip().split(',')
                if len(parts) < 8: continue
                try:
                    coords = [float(x) for x in parts[:8]]
                    poly = np.array(coords).reshape((4, 2))
                    text = ','.join(parts[8:])
                    tags.append(text == '###')
                    polys.append(poly)
                except: continue
        return np.array(polys) if polys else np.array([]), np.array(tags) if tags else np.array([])

    def shrink_polygon(self, poly):
        polygon = Polygon(poly)
        if polygon.area == 0 or polygon.length == 0: return poly
        d = polygon.area * (1 - self.shrink_ratio ** 2) / polygon.length
        clipper = pyclipper.PyclipperOffset()
        clipper.AddPath(poly.astype(int).tolist(), pyclipper.JT_ROUND, pyclipper.ET_CLOSEDPOLYGON)
        shrunk = clipper.Execute(-d)
        return np.array(shrunk[0]).reshape(-1, 2) if shrunk else poly

    def augment_image(self, img, polys):
        h, w = img.shape[:2]
        # Random rotation (-10 to 10 degrees)
        if random.random() > 0.5:
            angle = random.uniform(-10, 10)
            M = cv2.getRotationMatrix2D((w/2, h/2), angle, 1.0)
            img = cv2.warpAffine(img, M, (w, h))
            if len(polys) > 0:
                for i in range(len(polys)):
                    pts = np.hstack([polys[i], np.ones((len(polys[i]), 1))])
                    polys[i] = pts @ M.T
        # Random brightness/contrast
        if random.random() > 0.5:
            alpha = random.uniform(0.8, 1.2)
            beta = random.uniform(-20, 20)
            img = np.clip(alpha * img + beta, 0, 255).astype(np.uint8)
        # Random horizontal flip
        if random.random() > 0.5:
            img = cv2.flip(img, 1)
            if len(polys) > 0:
                for i in range(len(polys)):
                    polys[i][:, 0] = w - polys[i][:, 0]
        return img, polys

    def gen_maps(self, shape, polys, tags):
        h, w = shape
        prob = np.zeros((h, w), dtype=np.float32)
        mask = np.ones((h, w), dtype=np.float32)
        for i, poly in enumerate(polys):
            poly_int = np.round(poly).astype(np.int32)
            if tags[i]:
                cv2.fillPoly(mask, [poly_int], 0.0)
                continue
            shrunk = self.shrink_polygon(poly)
            cv2.fillPoly(prob, [np.round(shrunk).astype(np.int32)], 1.0)
        return prob, mask

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        name = os.path.basename(img_path).split('.')[0]
        gt_path = os.path.join(self.gt_dir, f'gt_{name}.txt')
        if not os.path.exists(gt_path):
            gt_path = os.path.join(self.gt_dir, f'{name}.txt')

        img = cv2.imread(img_path)
        if img is None:
            img = np.zeros((640, 640, 3), dtype=np.uint8)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        polys, tags = self.load_gt(gt_path)

        # Augmentation
        if self.augment and len(polys) > 0:
            img, polys = self.augment_image(img, polys)

        # Resize
        sx, sy = self.img_size[0] / w, self.img_size[1] / h
        img = cv2.resize(img, self.img_size)
        if len(polys) > 0:
            polys[:, :, 0] *= sx
            polys[:, :, 1] *= sy

        prob, mask = self.gen_maps(self.img_size, polys, tags)
        thresh = np.zeros_like(prob)
        thresh_mask = mask.copy()

        # Normalize
        img = img.astype(np.float32) / 255.0
        img[:,:,0] = (img[:,:,0] - 0.485) / 0.229
        img[:,:,1] = (img[:,:,1] - 0.456) / 0.224
        img[:,:,2] = (img[:,:,2] - 0.406) / 0.225

        return {
            'image': torch.from_numpy(img).permute(2,0,1),
            'prob_map': torch.from_numpy(prob).unsqueeze(0),
            'mask': torch.from_numpy(mask).unsqueeze(0),
            'thresh_map': torch.from_numpy(thresh).unsqueeze(0),
            'thresh_mask': torch.from_numpy(thresh_mask).unsqueeze(0),
        }

Overwriting det_dataset.py


## 4. Write Training Script

In [7]:
%%writefile train_det.py
import os, time, torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader
from tqdm import tqdm
from sadbnet_model import SADBNet
from det_dataset import DetDataset

BATCH_SIZE = 8
EPOCHS = 100
LR = 5e-5
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CKPT = '/kaggle/working/checkpoints'
TRAIN_IMG = '/kaggle/working/icdar2015/train_images'
TRAIN_GT = '/kaggle/working/icdar2015/train_gt'

class DBNetLoss(nn.Module):
    def __init__(self, alpha=1.0, beta=10.0, l1_scale=10.0):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.bce = nn.BCELoss(reduction='none')
        self.l1 = nn.L1Loss(reduction='none')
        self.l1_scale = l1_scale

    def forward(self, pred, target):
        prob, thresh, binary = pred
        t_prob = target['prob_map']
        mask = target['mask']
        t_thresh = target['thresh_map']
        t_mask = target['thresh_mask']
        bce = torch.sum(self.bce(prob, t_prob) * mask) / (torch.sum(mask) + 1e-6)
        l1 = torch.sum(self.l1(thresh, t_thresh) * t_mask) / (torch.sum(t_mask) + 1e-6) * self.l1_scale
        inter = torch.sum(binary * t_prob * mask)
        union = torch.sum(binary * mask) + torch.sum(t_prob * mask)
        dice = 1.0 - (2.0 * inter) / (union + 1e-6)
        total = bce + self.alpha * l1 + self.beta * dice
        return total, bce, l1, dice

def train():
    os.makedirs(CKPT, exist_ok=True)
    print('='*60)
    print('SA-DBNet v2.0 — Kaggle Training (100 Epochs)')
    print('='*60)

    ds = DetDataset(TRAIN_IMG, TRAIN_GT, augment=True)
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)

    model = SADBNet().to(DEVICE)
    # Use both T4 GPUs if available
    if torch.cuda.device_count() > 1:
        print(f'Using {torch.cuda.device_count()} GPUs!')
        model = nn.DataParallel(model)

    params = sum(p.numel() for p in model.parameters())
    print(f'Parameters: {params/1e6:.2f}M')

    criterion = DBNetLoss()
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

    best_loss = float('inf')

    for epoch in range(EPOCHS):
        model.train()
        total_loss, total_bce, total_dice = 0, 0, 0

        pbar = tqdm(loader, desc=f'Epoch {epoch+1}/{EPOCHS}')
        for batch in pbar:
            images = batch['image'].to(DEVICE)
            target = {k: v.to(DEVICE) for k, v in batch.items() if k != 'image'}

            optimizer.zero_grad()
            pred = model(images)
            loss, bce, l1, dice = criterion(pred, target)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()

            total_loss += loss.item()
            total_bce += bce.item()
            total_dice += dice.item()
            pbar.set_postfix(loss=f'{loss.item():.3f}', bce=f'{bce.item():.4f}')

        scheduler.step()
        avg = total_loss / len(loader)
        lr = optimizer.param_groups[0]['lr']
        print(f'Epoch {epoch+1} | Loss: {avg:.4f} | BCE: {total_bce/len(loader):.4f} | Dice: {total_dice/len(loader):.4f} | LR: {lr:.6f}')

        # Save
        state = model.module.state_dict() if hasattr(model, 'module') else model.state_dict()
        torch.save(state, os.path.join(CKPT, 'sadbnet_kaggle_latest.pth'))
        if avg < best_loss:
            best_loss = avg
            torch.save(state, os.path.join(CKPT, 'sadbnet_kaggle_best.pth'))
            print(f'  >>> New Best: {best_loss:.4f}')

    print(f'\nDone! Best Loss: {best_loss:.4f}')

if __name__ == '__main__':
    train()

Overwriting train_det.py


## 5. Train!

In [15]:
!sed -i "s|/kaggle/input/datasets/kagglemodeltraining/icdar-2015-detection/icdar2015/train_images|/kaggle/input/datasets/kagglemodeltraining/icdar-2015-detection/icdar2015_detection/icdar2015/train_images|" train_det.py
!sed -i "s|/kaggle/input/datasets/kagglemodeltraining/icdar-2015-detection/icdar2015/train_gt|/kaggle/input/datasets/kagglemodeltraining/icdar-2015-detection/icdar2015_detection/icdar2015/train_gt|" train_det.py
!python train_det.py


SA-DBNet — Fine-Tuning from 44% F1
DetDataset: 1000 images from /kaggle/input/datasets/kagglemodeltraining/icdar-2015-detection/icdar2015_detection/icdar2015/train_images
Loaded pre-trained 44% F1 weights!
Using 2 GPUs!
Parameters: 12.84M
Epoch 1/100: 100%|████| 125/125 [01:08<00:00,  1.82it/s, bce=0.0164, loss=4.264]
Epoch 1 | Loss: 4.3064 | BCE: 0.0111 | Dice: 0.2624 | LR: 0.000050
  >>> New Best: 4.3064
Epoch 2/100: 100%|████| 125/125 [01:05<00:00,  1.90it/s, bce=0.0201, loss=3.984]
Epoch 2 | Loss: 4.2289 | BCE: 0.0108 | Dice: 0.2552 | LR: 0.000050
  >>> New Best: 4.2289
Epoch 3/100: 100%|████| 125/125 [01:06<00:00,  1.89it/s, bce=0.0395, loss=6.414]
Epoch 3 | Loss: 4.2258 | BCE: 0.0109 | Dice: 0.2553 | LR: 0.000050
  >>> New Best: 4.2258
Epoch 4/100: 100%|████| 125/125 [01:06<00:00,  1.88it/s, bce=0.0060, loss=3.716]
Epoch 4 | Loss: 4.1327 | BCE: 0.0103 | Dice: 0.2456 | LR: 0.000050
  >>> New Best: 4.1327
Epoch 5/100: 100%|████| 125/125 [01:06<00:00,  1.88it/s, bce=0.0168, loss=4.3

## 6. Download Weights

In [13]:
import os
import zipfile
from IPython.display import FileLink

# 1. Name your output file
output_filename = "SA-DBNet v2.0_outputs.zip"

# 2. Create a zip of everything in the /kaggle/working directory
with zipfile.ZipFile(output_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk('/kaggle/working'):
        for file in files:
            if file != output_filename: # Don't zip the zip file itself
                zipf.write(os.path.join(root, file), os.path.relpath(os.path.join(root, file), '/kaggle/working'))

print("Zip created successfully!")

# 3. Create a clickable link to download it
FileLink(output_filename)

Zip created successfully!


/kaggle/working/SA-DBNet v2.0_outputs.zip